# Setting Up Your `climattr` Environment Locally

This notebook sets up Python on your **own machine** (Windows or Linux/macOS) and downloads only the data files needed for your chosen climate variable.

**Run this notebook once** before opening `02_WBGT_Attribution.ipynb`.

---

### What this notebook does

1. Guides you through creating a Python environment
2. Installs the `climattr` package and dependencies
3. Downloads only the data for **one variable** you choose
4. Prints the paths to paste into the analysis notebook

### Data server

All data is hosted at:
```
https://dataserver.cptec.inpe.br/dataserver_sazonal/cssp_summer_school/
```

There are two datasets:

| Dataset | Description | Files per variable |
|---------|-------------|-------------------|
| **ERA5** | Observational reanalysis (1979–2024) | ~46 files |
| **HadGEM3-A-N216** | Climate model ensembles (ALL + NAT forcing) | ~1,065 files |
| **utils** | Shapefiles + IPS vulnerability data | 6 files |

> **Download size estimate:** ~2–5 GB per variable. Make sure you have enough disk space before starting.

---
## Step 1 — Install Python (Anaconda or Miniconda)

> **Skip this step** if you already have Anaconda, Miniconda, or a Python 3.9+ installation on your machine.  
> You can check by opening a terminal and running `conda --version` or `python --version`.

**Miniconda** is recommended — it is a lightweight installer (~100 MB) that includes only Python and the `conda` package manager. Anaconda (~3 GB) bundles hundreds of extra scientific packages you may not need.

---

### Windows

1. Download the Miniconda Windows installer from:  
   **[https://docs.conda.io/en/latest/miniconda.html](https://docs.conda.io/en/latest/miniconda.html)**  
   → choose **"Miniconda3 Windows 64-bit"** (`.exe` file)

2. Run the `.exe` and follow the prompts:
   - Accept the licence
   - Install for **"Just Me"** (recommended)
   - On the *Advanced Options* screen, tick **"Add Miniconda3 to my PATH environment variable"**

3. After installation, open the **Anaconda Prompt** from the Start menu to use `conda`.

---

### Linux

Open a terminal and run:

```bash
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh
```

- Press `Enter` to scroll through the licence, then type `yes` to accept.
- Accept the default install location.
- When asked *"Do you wish the installer to initialize Miniconda3?"*, type `yes`.

Reload your shell:
```bash
source ~/.bashrc
```

---

### macOS

**Apple Silicon (M1 / M2 / M3):**
```bash
curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-arm64.sh
bash Miniconda3-latest-MacOSX-arm64.sh
```

**Intel Mac:**
```bash
curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-x86_64.sh
bash Miniconda3-latest-MacOSX-x86_64.sh
```

Follow the same prompts as Linux. Reload your shell afterwards:
```bash
source ~/.zshrc
```

Alternatively, download the graphical `.pkg` installer from  
**[https://docs.conda.io/en/latest/miniconda.html](https://docs.conda.io/en/latest/miniconda.html)** and double-click to install.

---

### Verify the installation (all platforms)

Open a **new** terminal (or Anaconda Prompt on Windows) and run:

```bash
conda --version
python --version
```

Both should print version numbers. If `conda` is not recognised, close and reopen your terminal and try again.

---
## Step 2 — Create a Python environment

Run these commands in a **terminal** (not inside this notebook).

On **Windows**, use the **Anaconda Prompt** from the Start menu.  
On **Linux / macOS**, use any terminal.

```bash
conda create -n climattr-env python=3.11 -y
conda activate climattr-env
pip install ipykernel
pip install git+https://github.com/Attribution-and-Impacts-Workshops/climattr@master
python -m ipykernel install --user --name=climattr-env
```

---

### Step 2b — Launch JupyterLab and select the kernel

With the environment activated, launch JupyterLab:
```bash
jupyter lab
```
Then in this notebook, click the kernel name in the top-right corner and select **"climattr-env"**.

> If **"climattr-env"** does not appear in the list, refresh the browser page and try again.

---
## Step 3 — Verify the environment

Make sure the **climattr-env** kernel is selected (top right), then run the cell below.

In [ ]:
import sys
import importlib

required = {
    'climattr':    'climattr',
    'xarray':      'xarray',
    'numpy':       'numpy',
    'matplotlib':  'matplotlib',
    'cartopy':     'cartopy',
    'scipy':       'scipy',
    'pandas':      'pandas',
    'geopandas':   'geopandas',
}

all_ok = True
for name, mod in required.items():
    try:
        m = importlib.import_module(mod)
        version = getattr(m, '__version__', 'installed')
        print(f'  ✓  {name:<14} {version}')
    except ImportError:
        print(f'  ✗  {name:<14} NOT FOUND — run: pip install {name}')
        all_ok = False

print()
if all_ok:
    print('All packages found. Ready to download data.')
else:
    print('Some packages are missing. Install them before continuing.')

---
## Step 4 — Configure: choose your variable

Set `VARIABLE` to the climate metric you will analyse, then run the cell.  
Only the data for that variable will be downloaded.

| `VARIABLE` | Full name | 
|------------|-----------|
| `'wbgt'`   | Wet Bulb Globe Temperature |
| `'cdd'`    | Cooling Degree Days |
| `'fwi'`    | Fire Weather Index |
| `'pr'`     | Precipitation (RX1day) | 
| `'spei1'`  | SPEI 1-month | 
| `'spei3'`  | SPEI 3-month | 

In [ ]:
from pathlib import Path

# ── USER SETTING ─────────────────────────────────────────────────────────────
# Set this to the variable you want to analyse.
# Options: 'wbgt', 'cdd', 'fwi', 'pr', 'spei1', 'spei3'
VARIABLE = 'wbgt'

# Root folder where data will be stored.
# Change this if you want to save data somewhere else.
# The structure data/era5/..., data/HadGEM3-A-N216/..., utils/ will be created here.
DATA_ROOT = Path('.').resolve()   # current directory (where this notebook lives)
# ─────────────────────────────────────────────────────────────────────────────

BASE_URL   = 'https://dataserver.cptec.inpe.br/dataserver_sazonal/cssp_summer_school'
MODEL_NAME = 'HadGEM3-A-N216'

VALID_VARS = ('wbgt', 'cdd', 'fwi', 'pr', 'spei1', 'spei3')
assert VARIABLE in VALID_VARS, (
    f"Unknown variable '{VARIABLE}'. Choose from: {VALID_VARS}"
)

# ── ERA5 file-naming rules per variable ──────────────────────────────────────
ERA5_CONFIG = {
    'wbgt':  {'prefix': 'wbgt-daily_max',  'single_file': False, 'nc_var': 'wbgt'},
    'cdd':   {'prefix': 'cdd-monthly_mean','single_file': False, 'nc_var': 'cdd'},
    'fwi':   {'prefix': 'fwi-daily_max',   'single_file': False, 'nc_var': 'fwi'},
    'pr':    {'prefix': 'tp-daily_sum',    'single_file': False, 'nc_var': 'pr'},
    'spei1': {'prefix': 'spei1',           'single_file': True,  'nc_var': 'spei1'},
    'spei3': {'prefix': 'spei3',           'single_file': True,  'nc_var': 'spei3'},
}
ERA5_YEARS = {
    'wbgt':  range(1979, 2025),
    'cdd':   range(1979, 2025),
    'fwi':   range(1979, 2025),
    'pr':    range(1940, 2025),
    'spei1': None,   # single file
    'spei3': None,
}

cfg = ERA5_CONFIG[VARIABLE]
print(f'Variable selected : {VARIABLE}')
print(f'Data root         : {DATA_ROOT}')
print(f'ERA5 nc_var       : {cfg["nc_var"]}')
if cfg['single_file']:
    print(f'ERA5 files        : 1 single file ({cfg["prefix"]}_1979-2024.nc)')
else:
    years = ERA5_YEARS[VARIABLE]
    print(f'ERA5 files        : {len(list(years))} annual files ({years.start}–{years.stop - 1})')

---
## Step 5 — Download ERA5 observational data

This cell downloads the ERA5 reanalysis files for your chosen variable.  
Files already present on disk are skipped automatically.

> ERA5 files are downloaded one year at a time. This may take **5–15 minutes** depending on your internet speed.

In [ ]:
import urllib.request
import urllib.error

def download_file(url, dest_path, label=''):
    """Download a single file with a basic progress indicator. Skips if already present."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)

    if dest_path.exists():
        print(f'  skip  {dest_path.name}')
        return False   # not downloaded (already existed)

    print(f'  ↓     {label or dest_path.name} … ', end='', flush=True)
    try:
        urllib.request.urlretrieve(url, dest_path)
        size_mb = dest_path.stat().st_size / 1024 / 1024
        print(f'{size_mb:.1f} MB')
        return True
    except urllib.error.HTTPError as e:
        print(f'ERROR {e.code}')
        return False
    except Exception as e:
        print(f'ERROR {e}')
        return False


# ── ERA5 download ─────────────────────────────────────────────────────────────
era5_folder = DATA_ROOT / 'data' / 'era5' / VARIABLE
era5_url    = f'{BASE_URL}/data/era5/{VARIABLE}'

cfg   = ERA5_CONFIG[VARIABLE]
years = ERA5_YEARS[VARIABLE]

print(f'Downloading ERA5 {VARIABLE} → {era5_folder}')
print()

downloaded = skipped = 0

if cfg['single_file']:
    # Single file covering the full period
    fname = f'{cfg["prefix"]}_1979-2024.nc'
    ok = download_file(f'{era5_url}/{fname}', era5_folder / fname)
    if ok: downloaded += 1
    else:  skipped   += 1
else:
    total = len(list(years))
    for i, year in enumerate(years, 1):
        fname = f'{cfg["prefix"]}-{year}.nc'
        ok = download_file(
            f'{era5_url}/{fname}',
            era5_folder / fname,
            label=f'[{i}/{total}] {fname}'
        )
        if ok: downloaded += 1
        else:  skipped    += 1

print()
print(f'ERA5 done — {downloaded} downloaded, {skipped} already on disk.')

---
## Step 6 — Download HadGEM3-A-N216 model ensembles

The attribution analysis uses three sets of model runs:

| Run | Forcing | Purpose | Files |
|-----|---------|---------|-------|
| `historical` | ALL historical forcings (1960–2013) | Bias correction & validation | ~15 |
| `historicalExt` | ALL forcings extended to 2024 | Factual world | ~525 |
| `historicalNatExt` | Natural-only forcings extended to 2024 | Counterfactual world | ~524 |

> This step downloads ~1,064 NetCDF files. Allow **30–90 minutes** depending on your internet speed.  
> Files already on disk are skipped, so you can safely re-run this cell after an interruption.

In [ ]:
import html.parser
import concurrent.futures
import threading

class _LinkParser(html.parser.HTMLParser):
    """Minimal HTML parser that extracts href values from anchor tags."""
    def __init__(self):
        super().__init__()
        self.links = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            for k, v in attrs:
                if k == 'href' and v and not v.startswith('?') and not v.startswith('/'):
                    self.links.append(v)


def list_remote_files(url):
    """Return list of .nc filenames available at a directory URL."""
    try:
        with urllib.request.urlopen(url) as resp:
            parser = _LinkParser()
            parser.feed(resp.read().decode('utf-8', errors='ignore'))
        return [l for l in parser.links if l.endswith('.nc')]
    except Exception as e:
        print(f'  WARNING: could not list {url}: {e}')
        return []


# ── Download all three model runs ────────────────────────────────────────────
RUNS = ['historical', 'historicalExt', 'historicalNatExt']
MAX_WORKERS = 8   # parallel downloads; reduce if you hit connection errors

_lock    = threading.Lock()
_counter = {'done': 0, 'skip': 0, 'err': 0}

def _worker(args):
    url, dest = args
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        with _lock:
            _counter['skip'] += 1
        return
    try:
        urllib.request.urlretrieve(url, dest)
        with _lock:
            _counter['done'] += 1
    except Exception:
        try:
            dest.unlink(missing_ok=True)
        except Exception:
            pass
        with _lock:
            _counter['err'] += 1


for run in RUNS:
    run_url    = f'{BASE_URL}/data/{MODEL_NAME}/{run}/{VARIABLE}'
    run_folder = DATA_ROOT / 'data' / MODEL_NAME / run / VARIABLE

    print(f'Listing {run}/{VARIABLE} … ', end='', flush=True)
    files = list_remote_files(run_url)
    print(f'{len(files)} files')

    tasks = [(f'{run_url}/{f}', run_folder / f) for f in files]

    _counter['done'] = _counter['skip'] = _counter['err'] = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(_worker, t) for t in tasks]
        for i, f in enumerate(concurrent.futures.as_completed(futures), 1):
            if i % 50 == 0 or i == len(tasks):
                print(
                    f'  {run}: {i}/{len(tasks)}  '
                    f'(downloaded={_counter["done"]}, '
                    f'skipped={_counter["skip"]}, '
                    f'errors={_counter["err"]})',
                    flush=True
                )

    if _counter['err']:
        print(f'  ⚠  {_counter["err"]} file(s) failed — re-run this cell to retry.')
    else:
        print(f'  ✓  {run} complete')
    print()

---
## Step 7 — Download utility files (shapefiles + IPS data)

These files are needed by `03_Vulnerability.ipynb` and the spatial masking in `02_WBGT_Attribution.ipynb`.  
They are small (~10 MB total) and download quickly.

In [ ]:
UTILS_URL    = f'{BASE_URL}/utils'
utils_folder = DATA_ROOT / 'utils'

# ── Shapefiles ────────────────────────────────────────────────────────────────
SHAPEFILES = {
    'BR_Municipios_2022': ['BR_Municipios_2022.cpg', 'BR_Municipios_2022.dbf',
                           'BR_Municipios_2022.prj', 'BR_Municipios_2022.shp',
                           'BR_Municipios_2022.shx'],
    'BR_UF_2022':         ['BR_UF_2022.cpg', 'BR_UF_2022.dbf',
                           'BR_UF_2022.prj', 'BR_UF_2022.shp',
                           'BR_UF_2022.shx'],
}

print('Downloading shapefiles...')
for folder_name, files in SHAPEFILES.items():
    dest_folder = utils_folder / folder_name
    for fname in files:
        download_file(
            f'{UTILS_URL}/{folder_name}/{fname}',
            dest_folder / fname
        )

# ── IPS vulnerability data ────────────────────────────────────────────────────
print()
print('Downloading IPS data...')
download_file(
    f'{UTILS_URL}/ips_brasil_municipios.csv',
    utils_folder / 'ips_brasil_municipios.csv'
)

print()
print('Utility files complete.')

---
## Step 8 — Verify the download

Check that all expected files are present on disk.

In [ ]:
import os

def count_files(folder, pattern='*.nc'):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len(list(folder.glob(pattern)))


def folder_size_mb(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(f.stat().st_size for f in folder.rglob('*') if f.is_file()) / 1024 / 1024


print(f'Variable: {VARIABLE}')
print(f'Data root: {DATA_ROOT}')
print()

checks = [
    ('ERA5 obs',                DATA_ROOT / 'data' / 'era5'         / VARIABLE),
    ('Model historical',        DATA_ROOT / 'data' / MODEL_NAME / 'historical'        / VARIABLE),
    ('Model historicalExt',     DATA_ROOT / 'data' / MODEL_NAME / 'historicalExt'     / VARIABLE),
    ('Model historicalNatExt',  DATA_ROOT / 'data' / MODEL_NAME / 'historicalNatExt'  / VARIABLE),
    ('Utils shapefiles',        DATA_ROOT / 'utils'),
]

all_ok = True
for label, folder in checks:
    n   = count_files(folder)
    mb  = folder_size_mb(folder)
    ok  = n > 0
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {label:<30}  {n:>5} files   {mb:>8.1f} MB   {folder}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('All directories populated. Proceed to Step 8.')
else:
    print('Some directories are empty. Re-run the download cells above.')

---
## Step 9 — Path reference for the analysis notebook

Run the cell below to generate the path strings you need to paste into `02_WBGT_Attribution.ipynb` (the **File paths** cell near the top).

> On Windows, Python uses forward slashes (`/`) in glob patterns so the paths printed below work on all platforms.

In [ ]:
# Convert to forward-slash strings for use in glob patterns
def fwd(p):
    return str(p).replace('\\', '/')

era5_dir  = DATA_ROOT / 'data' / 'era5'         / VARIABLE
all_dir   = DATA_ROOT / 'data' / MODEL_NAME / 'historicalExt'     / VARIABLE
nat_dir   = DATA_ROOT / 'data' / MODEL_NAME / 'historicalNatExt'  / VARIABLE
hist_dir  = DATA_ROOT / 'data' / MODEL_NAME / 'historical'        / VARIABLE
mun_shp   = DATA_ROOT / 'utils' / 'BR_Municipios_2022' / 'BR_Municipios_2022.shp'
uf_shp    = DATA_ROOT / 'utils' / 'BR_UF_2022' / 'BR_UF_2022.shp'
ips_csv   = DATA_ROOT / 'utils' / 'ips_brasil_municipios.csv'

# ── ERA5 pattern differs per variable ────────────────────────────────────────
era5_prefix = ERA5_CONFIG[VARIABLE]['prefix']
if ERA5_CONFIG[VARIABLE]['single_file']:
    obs_pattern = f'{fwd(era5_dir)}/{VARIABLE}_1979-2024.nc'
else:
    obs_pattern = f'{fwd(era5_dir)}/{era5_prefix}-*.nc'

hist_pattern = f'{fwd(hist_dir)}/{VARIABLE}_*_{MODEL_NAME}_historical_*_19600101-20131230.nc'
all_pattern  = f'{fwd(all_dir)}/{VARIABLE}_*_{MODEL_NAME}_historicalExt_*'
nat_pattern  = f'{fwd(nat_dir)}/{VARIABLE}_*_{MODEL_NAME}_historicalNatExt_*'

print('─' * 70)
print('Copy these lines into the "File paths" cell of 02_WBGT_Attribution.ipynb')
print('─' * 70)
print()
print(f'OBS_PATH  = \'{obs_pattern}\'')
print(f'ALL_PATH  = \'{all_pattern}\'')
print(f'NAT_PATH  = \'{nat_pattern}\'')
print(f'HIST_PATH = \'{hist_pattern}\'')
print()
print('─' * 70)
print('For 03_Vulnerability.ipynb:')
print('─' * 70)
print()
print(f'IPS_PATH = \'{fwd(ips_csv)}\'')
print(f'MUN_SHP  = \'{fwd(mun_shp)}\'')
print(f'UF_SHP   = \'{fwd(uf_shp)}\'')
print()
print('─' * 70)
print()
print(f'VARIABLE = \'{ERA5_CONFIG[VARIABLE]["nc_var"]}\'')

---
## Step 10 — Final check: test a data load

Run the cell below to open one ERA5 file and confirm the data can be read correctly.

In [ ]:
import xarray as xr

era5_files = sorted((DATA_ROOT / 'data' / 'era5' / VARIABLE).glob('*.nc'))

if not era5_files:
    print('No ERA5 files found. Run the download cells first.')
else:
    test_file = era5_files[-1]   # most recent year
    print(f'Opening: {test_file.name}')
    ds = xr.open_dataset(test_file)
    print(ds)
    print()
    nc_var = ERA5_CONFIG[VARIABLE]['nc_var']
    if nc_var in ds:
        da = ds[nc_var]
        print(f'Variable  : {nc_var}')
        print(f'Shape     : {da.shape}')
        print(f'Dims      : {da.dims}')
        print(f'Time range: {str(da.time.values[[0, -1]])}')
        print()
        print('Data loads correctly. You are ready to run 02_WBGT_Attribution.ipynb.')
    else:
        available = list(ds.data_vars)
        print(f'Variable "{nc_var}" not found. Available variables: {available}')
        print(f'Update VARIABLE in the analysis notebook to match one of the above.')
    ds.close()